# Demo — Crawl link tin tức năm (2023, 2024 , ...) bằng `gnews` và xuất Word + Excel (Python 3.11)

Mục tiêu: lấy danh sách **link bài viết** về các ngân hàng (**BIDV, VietinBank, Vietcombank, Agribank , ...**) bằng `gnews` (query theo `site:`), sau đó **xuất thẳng link** ra:
- File Word (`.docx`)
- File Excel (`.xlsx`)

Yêu cầu: chỉ crawl các bài trong **năm 2023** ( chỉ định năm cần craw) và export. 

Nguồn crawl (dùng `site:`):
- `tapchinganhang.gov.vn`
- `vnexpress.net` (Kinh doanh)
- `baochinhphu.vn`
- `cafef.vn`
- `fili.vn`
- `thitruongtaichinhtiente.vn` (nếu domain thực tế khác, sửa trong biến `TARGET_SITES`)



## 1) Cài đặt & import thư viện

In [1]:
# Nếu bạn chưa cài package trong kernel hiện tại, hãy chạy cell này.
# (Trong VS Code Notebook, lệnh pip sẽ cài vào đúng kernel nếu bạn đã chọn Python env.)

%pip -q install gnews googlenewsdecoder python-docx pandas python-dateutil openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import logging
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable
from urllib.parse import urlparse, parse_qs, unquote

import pandas as pd
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.opc.constants import RELATIONSHIP_TYPE
from gnews import GNews
from googlenewsdecoder import gnewsdecoder

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("gnews-demo")

## 2) Cấu hình: ngân hàng, domain nguồn, tham số truy vấn

In [3]:
# ====== Tham số bạn có thể chỉnh nhanh ======
YEAR = 2023
START_DATE = f"{YEAR}-01-01"
END_DATE = f"{YEAR + 1}-01-01"

# Mục tiêu: CHỈ lấy tin trong năm YEAR.
# - Query sẽ có after/before để bias đúng năm
# - Sau khi crawl, sẽ lọc cứng theo cột published để không lẫn năm khác
MAX_RESULTS_PER_QUERY = 1000
SLEEP_SECONDS = 0.8  # giảm rủi ro bị rate-limit

# Project root: Lưu thư mục data/ trực tiếp trong Thread_1
PROJECT_ROOT = Path.cwd()

# Output base: data/<YEAR> (sẽ tự động thêm thư mục <BANK_NAME>/bronze ở các bước sau)
DATA_DIR = PROJECT_ROOT / "data" / str(YEAR)
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
STAGE_DIRNAME = "bronze"

# ====== Ngân hàng ======
BANKS: dict[str, list[str]] = {
    "BIDV": ["BIDV", "Ngân hàng BIDV" , "Ngân hàng TMCP Đầu tư và Phát triển Việt Nam" ],
    # "VietinBank": ["VietinBank", "Vietin bank", "Ngân hàng Công Thương"],
    # "Vietcombank": ["Vietcombank", "Vietcom bank", "Ngân hàng Ngoại thương"],
    # "Agribank": ["Agribank", "Ngân hàng Nông nghiệp", "Ngân hàng NN&PTNT"],
}

# ====== Domain mục tiêu (để tạo query site:) ======
# Không dùng để lọc, chỉ dùng để tạo query.
TARGET_SITES: dict[str, str] = {
    "tapchinganhang.gov.vn": "Tạp chí Ngân hàng",
    "vnexpress.net": "VnExpress (Kinh doanh)",
    "baochinhphu.vn": "Báo điện tử Chính phủ",
    "cafef.vn": "CafeF",
    "fili.vn": "FiLi",
    "thitruongtaichinhtiente.vn": "Thị trường TC-TT",
}

# ====== Cấu hình gnews ======
GNEWS_LANG = "vi"
GNEWS_COUNTRY = "VN"
# QUAN TRỌNG: không set period (when:365d) vì sẽ kéo kết quả về 2025/2026
GNEWS_PERIOD = None

## 3) Tạo câu truy vấn GNews theo ngân hàng + nguồn (site:)

In [4]:
def build_queries(banks: dict[str, list[str]], hosts: Iterable[str]) -> list[dict]:
    """Tạo danh sách query theo ngân hàng × site.

    Query mẫu: ("BIDV" OR "Ngân hàng BIDV") site:vnexpress.net after:2023-01-01 before:2024-01-01
    """
    queries: list[dict] = []
    time_clause = f"after:{START_DATE} before:{END_DATE}"
    for bank_name, bank_aliases in banks.items():
        alias_part = " OR ".join([f'\"{a}\"' for a in bank_aliases])
        bank_clause = f"({alias_part})"
        context_clause = "(ngân hàng OR bank)"
        for host in hosts:
            q = f"{bank_clause} {context_clause} site:{host} {time_clause}"
            # FIX: gnews._get_news_more_than_100 không encode space -> tự thay space bằng %20 (như gnews.get_news làm)
            q = "%20".join(q.split())
            queries.append({"bank": bank_name, "host": host, "query": q})
    return queries

QUERY_SPECS = build_queries(BANKS, TARGET_SITES.keys())
len(QUERY_SPECS), QUERY_SPECS[0]

(6,
 {'bank': 'BIDV',
  'host': 'tapchinganhang.gov.vn',
  'query': '("BIDV"%20OR%20"Ngân%20hàng%20BIDV"%20OR%20"Ngân%20hàng%20TMCP%20Đầu%20tư%20và%20Phát%20triển%20Việt%20Nam")%20(ngân%20hàng%20OR%20bank)%20site:tapchinganhang.gov.vn%20after:2023-01-01%20before:2024-01-01'})

## 4) Thu thập bài viết bằng `gnews`

In [5]:
@dataclass
class RawNewsItem:
    bank: str
    host_target: str
    query: str
    title: str | None
    url: str | None
    published: str | None
    publisher: str | None
    description: str | None


def init_gnews() -> GNews:
    g = GNews(
        language=GNEWS_LANG,
        country=GNEWS_COUNTRY,
        period=GNEWS_PERIOD,
        max_results=MAX_RESULTS_PER_QUERY,
    )
    # Một số phiên bản gnews có các thuộc tính khác nhau; để mặc định cho an toàn.
    return g


def fetch_news_for_query(g: GNews, bank: str, host: str, query: str) -> list[RawNewsItem]:
    try:
        results = g.get_news(query) or []
    except Exception as e:
        logger.warning("Fetch failed for %s | %s: %s", bank, host, e)
        return []

    items: list[RawNewsItem] = []
    for r in results:
        # keys thường gặp: title, url, published date, publisher, description
        items.append(
            RawNewsItem(
                bank=bank,
                host_target=host,
                query=query,
                title=r.get("title"),
                url=r.get("url"),
                published=r.get("published date") or r.get("published") or r.get("date"),
                publisher=(r.get("publisher") or {}).get("title") if isinstance(r.get("publisher"), dict) else r.get("publisher"),
                description=r.get("description"),
            )
        )
    return items


gnews = init_gnews()

raw_items: list[RawNewsItem] = []
for i, spec in enumerate(QUERY_SPECS, start=1):
    bank = spec["bank"]
    host = spec["host"]
    query = spec["query"]

    logger.info("[%d/%d] %s | %s", i, len(QUERY_SPECS), bank, host)
    raw_items.extend(fetch_news_for_query(gnews, bank, host, query))
    time.sleep(SLEEP_SECONDS)

logger.info("Raw items (before year filter): %d", len(raw_items))
raw_df_all = pd.DataFrame(
    [item.__dict__ for item in raw_items],
    columns=list(RawNewsItem.__dataclass_fields__.keys()),
)
if raw_df_all.empty:
    logger.warning("Raw items = 0 (raw_df rỗng). Hãy kiểm tra lại query.")

# ---- Lọc cứng chỉ giữ đúng năm YEAR theo published ----
from datetime import timezone
from email.utils import parsedate_to_datetime


def _parse_published_dt(value: str | None) -> pd.Timestamp | pd.NaT:
    if not value or not isinstance(value, str):
        return pd.NaT
    try:
        dt = parsedate_to_datetime(value)
    except Exception:
        return pd.NaT
    if dt is None:
        return pd.NaT
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    ts = pd.Timestamp(dt)
    if ts.tz is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")
    return ts


raw_df_all["published_dt"] = raw_df_all["published"].apply(_parse_published_dt)
year_start = pd.Timestamp(f"{YEAR}-01-01", tz="UTC")
year_end = pd.Timestamp(f"{YEAR + 1}-01-01", tz="UTC")
mask_year = (raw_df_all["published_dt"].notna()) & (raw_df_all["published_dt"] >= year_start) & (raw_df_all["published_dt"] < year_end)
raw_df = raw_df_all.loc[mask_year].copy()

parsed_n = int(raw_df_all["published_dt"].notna().sum())
in_year_n = int(mask_year.sum())
dropped_n = len(raw_df_all) - in_year_n
print(f"Parsed published_dt: {parsed_n}/{len(raw_df_all)}")
print(f"In YEAR={YEAR}: {in_year_n} (dropped {dropped_n} not-in-year or unparsed)")

raw_df.head(10)

04/04/2026 04:06:29 PM - [1/6] BIDV | tapchinganhang.gov.vn
04/04/2026 04:06:36 PM - [2/6] BIDV | vnexpress.net
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\gnews\gnews.py:218: UserWarning: Only searches using get_news support date ranges. Start and end dates will be ignored.
  return self._get_news_more_than_100(key)
04/04/2026 04:07:01 PM - [3/6] BIDV | baochinhphu.vn
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\gnews\gnews.py:230: UserWarning: Searches for over 100 articles ignore date ranges.
  warnings.warn("Searches for over 100 articles ignore date ranges.", category=UserWarning)
04/04/2026 04:07:29 PM - [4/6] BIDV | cafef.vn
04/04/2026 04:07:54 PM - [5/6] BIDV | fili.vn
04/04/2026 04:07:59 PM - [6/6] BIDV | thitruongtaichinhtiente.vn
04/04/2026 04:08:24 PM - Raw items (before year filter): 475


Parsed published_dt: 475/475
In YEAR=2023: 432 (dropped 43 not-in-year or unparsed)


,bank,host_target,query,title,url,published,publisher,description,published_dt
0,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",Các nhân tố ảnh hưởng tới sự trung thành với n...,https://news.google.com/rss/articles/CBMi_wFBV...,"Tue, 19 Sep 2023 07:00:00 GMT",Tạp chí Ngân hàng,Các nhân tố ảnh hưởng tới sự trung thành với n...,2023-09-19 07:00:00+00:00
1,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",Hành trình chuyển đổi BIDV Core Banking và nhữ...,https://news.google.com/rss/articles/CBMivAFBV...,"Tue, 10 Oct 2023 07:00:00 GMT",Tạp chí Ngân hàng,Hành trình chuyển đổi BIDV Core Banking và nhữ...,2023-10-10 07:00:00+00:00
2,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",BIDV miễn phí hàng triệu tài khoản số đẹp thế ...,https://news.google.com/rss/articles/CBMilwFBV...,"Fri, 10 Nov 2023 08:00:00 GMT",Tạp chí Ngân hàng,BIDV miễn phí hàng triệu tài khoản số đẹp thế ...,2023-11-10 08:00:00+00:00
3,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",BIDV là ngân hàng đầu tiên ký hợp đồng tín dụn...,https://news.google.com/rss/articles/CBMi5AFBV...,"Wed, 05 Jul 2023 07:00:00 GMT",Tạp chí Ngân hàng,BIDV là ngân hàng đầu tiên ký hợp đồng tín dụn...,2023-07-05 07:00:00+00:00
4,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",BIDV trên hành trình trở thành “ngân hàng xanh...,https://news.google.com/rss/articles/CBMijgFBV...,"Tue, 31 Oct 2023 07:00:00 GMT",Tạp chí Ngân hàng,BIDV trên hành trình trở thành “ngân hàng xanh...,2023-10-31 07:00:00+00:00
5,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",BIDV triển khai chuẩn tin điện ISO20022 tại hệ...,https://news.google.com/rss/articles/CBMitgFBV...,"Wed, 11 Oct 2023 07:00:00 GMT",Tạp chí Ngân hàng,BIDV triển khai chuẩn tin điện ISO20022 tại hệ...,2023-10-11 07:00:00+00:00
6,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...","Thực thi ESG tại BIDV: Thuận lợi, khó khăn và ...",https://news.google.com/rss/articles/CBMirgFBV...,"Sat, 23 Dec 2023 08:00:00 GMT",Tạp chí Ngân hàng,"Thực thi ESG tại BIDV: Thuận lợi, khó khăn và ...",2023-12-23 08:00:00+00:00
7,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",Một số khuyến nghị nâng cao hiệu quả hoạt động...,https://news.google.com/rss/articles/CBMi9gFBV...,"Mon, 05 Jun 2023 07:00:00 GMT",Tạp chí Ngân hàng,Một số khuyến nghị nâng cao hiệu quả hoạt động...,2023-06-05 07:00:00+00:00
8,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",Thực thi ESG trong ngành Ngân hàng - Xu hướng ...,https://news.google.com/rss/articles/CBMitAFBV...,"Fri, 22 Dec 2023 08:00:00 GMT",Tạp chí Ngân hàng,Thực thi ESG trong ngành Ngân hàng - Xu hướng ...,2023-12-22 08:00:00+00:00
9,BIDV,tapchinganhang.gov.vn,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng...",Basel III: Quá trình thực hiện tại Việt Nam và...,https://news.google.com/rss/articles/CBMipwFBV...,"Mon, 27 Mar 2023 07:00:00 GMT",Tạp chí Ngân hàng,Basel III: Quá trình thực hiện tại Việt Nam và...,2023-03-27 07:00:00+00:00


## 5) Giải mã link Google News → link bài gốc + thống kê số bài

In [6]:
# Mục tiêu cell này: chỉ "crawl" (lấy danh sách bài) + decode link Google News -> link bài gốc

_decode_cache: dict[str, str] = {}

# Giảm rủi ro bị Google chặn (429) khi decode quá nhiều link liên tiếp
DECODE_MIN_INTERVAL_SECONDS = 1.2
DECODE_BLOCK_SECONDS_ON_429 = 30 * 60  # 30 phút
_last_decode_ts = 0.0
_decoder_blocked_until: float | None = None
_decoder_block_reason: str | None = None


def _strip_trailing_punct(url: str) -> str:
    # Word đôi khi bấm link kèm dấu câu => thành URL lỗi
    return url.strip().rstrip(")].,;")


def _extract_direct_from_google_redirect(url: str) -> str:
    """Nếu URL là dạng redirect của Google (google.com/url?...), bóc ra URL đích thật."""
    try:
        p = urlparse(url)
    except Exception:
        return url

    host = (p.netloc or "").lower()
    if not host.endswith("google.com"):
        return url

    qs = parse_qs(p.query)
    for key in ("url", "q", "continue"):
        if key in qs and qs[key]:
            candidate = unquote(qs[key][0]).strip()
            if candidate.startswith("http://") or candidate.startswith("https://"):
                return candidate

    return url


def _news_rss_to_articles(url: str) -> str:
    """Fallback: đổi link RSS (/rss/articles/...) sang link UI (/articles/...) để bấm dễ hơn."""
    try:
        p = urlparse(url)
        host = (p.netloc or "").lower()
        if host != "news.google.com":
            return url

        token = p.path.rsplit("/", 1)[-1]
        if not token:
            return url

        ceid = f"{GNEWS_COUNTRY}:{GNEWS_LANG}"
        return f"https://news.google.com/articles/{token}?hl={GNEWS_LANG}&gl={GNEWS_COUNTRY}&ceid={ceid}"
    except Exception:
        return url


def _rate_limit_decode_calls() -> None:
    global _last_decode_ts
    now = time.time()
    wait = DECODE_MIN_INTERVAL_SECONDS - (now - _last_decode_ts)
    if wait > 0:
        time.sleep(wait)
    _last_decode_ts = time.time()


def decode_url(url: str | None) -> str | None:
    """Tạo URL để click trong Word/Excel.

    - Nếu là news.google.com/rss/articles/... => cố gắng decode ra link bài gốc.
    - Nếu kết quả là google redirect (google.com/url?...), bóc tiếp ra link đích.

    Lưu ý: nếu Google trả 429 Too Many Requests, notebook sẽ tạm "dừng decode"
    trong một khoảng thời gian để tránh bị chặn nặng hơn.
    """
    global _decoder_blocked_until, _decoder_block_reason

    if not url:
        return None

    url = url.strip()
    if url in _decode_cache:
        return _decode_cache[url]

    out = url

    # Nếu đang bị block 429 thì bỏ qua decode, chỉ làm sạch + fallback sang /articles/
    if _decoder_blocked_until is not None and time.time() < _decoder_blocked_until:
        out = _news_rss_to_articles(out)
        out = _strip_trailing_punct(_extract_direct_from_google_redirect(out))
        _decode_cache[url] = out
        return out

    try:
        host = urlparse(url).netloc.lower()
    except Exception:
        host = ""

    if host == "news.google.com":
        try:
            _rate_limit_decode_calls()
            decoded = gnewsdecoder(url)
            if isinstance(decoded, dict) and decoded.get("status") and decoded.get("decoded_url"):
                out = str(decoded["decoded_url"]).strip()
            else:
                msg = str(decoded.get("message", "")) if isinstance(decoded, dict) else ""
                if "429" in msg or "Too Many Requests" in msg:
                    _decoder_block_reason = msg
                    _decoder_blocked_until = time.time() + DECODE_BLOCK_SECONDS_ON_429
                    out = _news_rss_to_articles(out)
                else:
                    out = _news_rss_to_articles(out)
        except Exception as e:
            # Không phá pipeline nếu decode fail
            _decoder_block_reason = repr(e)
            out = _news_rss_to_articles(out)

    # Nếu vẫn là redirect của google thì bóc ra URL đích
    out = _extract_direct_from_google_redirect(out)

    # Vá một số URL hay bị dính dấu câu cuối
    out = _strip_trailing_punct(out)

    _decode_cache[url] = out
    return out


articles = raw_df.copy()
if "url" not in articles.columns:
    print("WARNING: raw_df không có cột 'url' (có thể do crawl ra 0 bài).")
    articles["url_out"] = None
else:
    articles["url_out"] = articles["url"].apply(decode_url)

TOTAL_ARTICLES = len(articles)
print("Tổng số bài (raw, chưa dedupe):", TOTAL_ARTICLES)

# Thống kê nhanh theo ngân hàng và site target
summary = (
    articles.groupby(["bank", "host_target"], dropna=False)
    .size()
    .reset_index(name="n_articles")
    .sort_values(["bank", "n_articles"], ascending=[True, False])
)

display(summary)
display(articles[["bank", "host_target", "published", "title", "url_out"]].head(10))

# Xem còn bao nhiêu link vẫn là google (để biết mức độ decode thành công)
try:
    host_out = articles["url_out"].fillna("").apply(lambda u: urlparse(u).netloc.lower() if u else "")
    still_google = int(host_out.str.endswith("google.com").sum())
    print("Links still on *.google.com:", still_google)
except Exception:
    still_google = None

if _decoder_blocked_until is not None and time.time() < _decoder_blocked_until:
    print("NOTE: Google is rate-limiting decode (429). Hãy đợi ~30 phút rồi chạy lại cell này.")
    if _decoder_block_reason:
        print("Reason:", _decoder_block_reason[:250])

Tổng số bài (raw, chưa dedupe): 432


,bank,host_target,n_articles
0,BIDV,baochinhphu.vn,100
4,BIDV,thitruongtaichinhtiente.vn,99
5,BIDV,vnexpress.net,97
1,BIDV,cafef.vn,95
3,BIDV,tapchinganhang.gov.vn,25
2,BIDV,fili.vn,16


,bank,host_target,published,title,url_out
0,BIDV,tapchinganhang.gov.vn,"Tue, 19 Sep 2023 07:00:00 GMT",Các nhân tố ảnh hưởng tới sự trung thành với n...,https://news.google.com/articles/CBMi_wFBVV95c...
1,BIDV,tapchinganhang.gov.vn,"Tue, 10 Oct 2023 07:00:00 GMT",Hành trình chuyển đổi BIDV Core Banking và nhữ...,https://news.google.com/articles/CBMivAFBVV95c...
2,BIDV,tapchinganhang.gov.vn,"Fri, 10 Nov 2023 08:00:00 GMT",BIDV miễn phí hàng triệu tài khoản số đẹp thế ...,https://news.google.com/articles/CBMilwFBVV95c...
3,BIDV,tapchinganhang.gov.vn,"Wed, 05 Jul 2023 07:00:00 GMT",BIDV là ngân hàng đầu tiên ký hợp đồng tín dụn...,https://news.google.com/articles/CBMi5AFBVV95c...
4,BIDV,tapchinganhang.gov.vn,"Tue, 31 Oct 2023 07:00:00 GMT",BIDV trên hành trình trở thành “ngân hàng xanh...,https://news.google.com/articles/CBMijgFBVV95c...
5,BIDV,tapchinganhang.gov.vn,"Wed, 11 Oct 2023 07:00:00 GMT",BIDV triển khai chuẩn tin điện ISO20022 tại hệ...,https://news.google.com/articles/CBMitgFBVV95c...
6,BIDV,tapchinganhang.gov.vn,"Sat, 23 Dec 2023 08:00:00 GMT","Thực thi ESG tại BIDV: Thuận lợi, khó khăn và ...",https://news.google.com/articles/CBMirgFBVV95c...
7,BIDV,tapchinganhang.gov.vn,"Mon, 05 Jun 2023 07:00:00 GMT",Một số khuyến nghị nâng cao hiệu quả hoạt động...,https://news.google.com/articles/CBMi9gFBVV95c...
8,BIDV,tapchinganhang.gov.vn,"Fri, 22 Dec 2023 08:00:00 GMT",Thực thi ESG trong ngành Ngân hàng - Xu hướng ...,https://news.google.com/articles/CBMitAFBVV95c...
9,BIDV,tapchinganhang.gov.vn,"Mon, 27 Mar 2023 07:00:00 GMT",Basel III: Quá trình thực hiện tại Việt Nam và...,https://news.google.com/articles/CBMipwFBVV95c...


Links still on *.google.com: 432
NOTE: Google is rate-limiting decode (429). Hãy đợi ~30 phút rồi chạy lại cell này.
Reason: Request error in decode_url: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://news.google.com/_/DotsSplashUi/data/batchexecute&q=EgR0YC_DGImqw84GIjAay53u968zoui-upafCmFpw96l6yvRSx5MlyLweUWHsKUNUZSqfrKCw


## 6) Chuẩn bị bảng export 

In [7]:
# Chuẩn bị bảng export (không dedupe).
# Lưu ý: raw_df đã được lọc cứng để chỉ còn bài trong YEAR theo published.

final_df = articles.copy()

# Các cột chính để export
export_cols = [
    "bank",
    "host_target",
    "published",
    "title",
    "url_out",
    "publisher",
    "query",
]
final_df_export = final_df[[c for c in export_cols if c in final_df.columns]].copy()

print("Số bài sẽ export:", len(final_df_export))
final_df_export.head(10)

Số bài sẽ export: 432


,bank,host_target,published,title,url_out,publisher,query
0,BIDV,tapchinganhang.gov.vn,"Tue, 19 Sep 2023 07:00:00 GMT",Các nhân tố ảnh hưởng tới sự trung thành với n...,https://news.google.com/articles/CBMi_wFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
1,BIDV,tapchinganhang.gov.vn,"Tue, 10 Oct 2023 07:00:00 GMT",Hành trình chuyển đổi BIDV Core Banking và nhữ...,https://news.google.com/articles/CBMivAFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
2,BIDV,tapchinganhang.gov.vn,"Fri, 10 Nov 2023 08:00:00 GMT",BIDV miễn phí hàng triệu tài khoản số đẹp thế ...,https://news.google.com/articles/CBMilwFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
3,BIDV,tapchinganhang.gov.vn,"Wed, 05 Jul 2023 07:00:00 GMT",BIDV là ngân hàng đầu tiên ký hợp đồng tín dụn...,https://news.google.com/articles/CBMi5AFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
4,BIDV,tapchinganhang.gov.vn,"Tue, 31 Oct 2023 07:00:00 GMT",BIDV trên hành trình trở thành “ngân hàng xanh...,https://news.google.com/articles/CBMijgFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
5,BIDV,tapchinganhang.gov.vn,"Wed, 11 Oct 2023 07:00:00 GMT",BIDV triển khai chuẩn tin điện ISO20022 tại hệ...,https://news.google.com/articles/CBMitgFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
6,BIDV,tapchinganhang.gov.vn,"Sat, 23 Dec 2023 08:00:00 GMT","Thực thi ESG tại BIDV: Thuận lợi, khó khăn và ...",https://news.google.com/articles/CBMirgFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
7,BIDV,tapchinganhang.gov.vn,"Mon, 05 Jun 2023 07:00:00 GMT",Một số khuyến nghị nâng cao hiệu quả hoạt động...,https://news.google.com/articles/CBMi9gFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
8,BIDV,tapchinganhang.gov.vn,"Fri, 22 Dec 2023 08:00:00 GMT",Thực thi ESG trong ngành Ngân hàng - Xu hướng ...,https://news.google.com/articles/CBMitAFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."
9,BIDV,tapchinganhang.gov.vn,"Mon, 27 Mar 2023 07:00:00 GMT",Basel III: Quá trình thực hiện tại Việt Nam và...,https://news.google.com/articles/CBMipwFBVV95c...,Tạp chí Ngân hàng,"(""BIDV""%20OR%20""Ngân%20hàng%20BIDV""%20OR%20""Ng..."


## 7) Xuất link ra Word + Excel

In [8]:
def add_hyperlink(paragraph, text: str, url: str):
    """Thêm hyperlink vào paragraph (python-docx không có API hyperlink trực tiếp)."""
    part = paragraph.part
    r_id = part.relate_to(url, RELATIONSHIP_TYPE.HYPERLINK, is_external=True)

    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), r_id)

    new_run = OxmlElement("w:r")
    rPr = OxmlElement("w:rPr")

    # style link (màu xanh + underline)
    c = OxmlElement("w:color")
    c.set(qn("w:val"), "0000FF")
    rPr.append(c)
    u = OxmlElement("w:u")
    u.set(qn("w:val"), "single")
    rPr.append(u)

    new_run.append(rPr)
    text_elm = OxmlElement("w:t")
    text_elm.text = text
    new_run.append(text_elm)
    hyperlink.append(new_run)

    paragraph._p.append(hyperlink)


def export_docx(df: pd.DataFrame, out_path: Path) -> Path:
    doc = Document()
    doc.add_heading("GNews links", level=0)
    doc.add_paragraph(f"Generated: {datetime.now().isoformat(timespec='seconds')}")
    doc.add_paragraph(f"Total articles: {len(df)}")

    for bank, bank_df in df.groupby("bank", sort=False):
        doc.add_heading(bank, level=1)

        for host, host_df in bank_df.groupby("host_target", sort=False):
            label = TARGET_SITES.get(host, host)
            doc.add_heading(f"{label} ({host})", level=2)

            for _, row in host_df.iterrows():
                title = (row.get("title") or "(no title)").strip()
                url = row.get("url_out") or row.get("url")
                published = row.get("published")

                p = doc.add_paragraph(style="List Bullet")
                if published:
                    p.add_run(f"[{published}] ")

                if url:
                    add_hyperlink(p, title, url)
                else:
                    p.add_run(title)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(str(out_path))
    return out_path


# Export Word + Excel THEO TỪNG NGÂN HÀNG (Cấu trúc: data/<YEAR>/<BANK_NAME>/bronze)
print("Total articles to export:", len(final_df_export))

# Lặp qua từng tên ngân hàng có trong danh sách thu thập
for bank_name in final_df_export["bank"].unique():
    bank_df = final_df_export[final_df_export["bank"] == bank_name]
    
    # Tạo folder cho riêng ngân hàng đó: data/2023/BIDV/bronze/
    bank_out_dir = DATA_DIR / bank_name / STAGE_DIRNAME
    bank_out_dir.mkdir(parents=True, exist_ok=True)
    
    out_docx = bank_out_dir / f"{bank_name}_news_gnews_{YEAR}_{TIMESTAMP}.docx"
    out_xlsx = bank_out_dir / f"{bank_name}_news_gnews_{YEAR}_{TIMESTAMP}.xlsx"
    
    # Lưu file
    export_docx(bank_df, out_docx)
    bank_df.to_excel(out_xlsx, index=False)
    
    print(f"[{bank_name}] Đã lưu DOCX: {out_docx}")
    print(f"[{bank_name}] Đã lưu XLSX: {out_xlsx}")

Total articles to export: 432
[BIDV] Đã lưu DOCX: d:\NCKH\thread_1\data\2023\BIDV\bronze\BIDV_news_gnews_2023_20260404_1606.docx
[BIDV] Đã lưu XLSX: d:\NCKH\thread_1\data\2023\BIDV\bronze\BIDV_news_gnews_2023_20260404_1606.xlsx


## 8) Thống kê số bài (theo ngân hàng/nguồn)

In [9]:
# Thống kê cuối cùng (theo yêu cầu: chỉ thống kê số bài)
print("Tổng số bài export:", len(final_df_export))

stats = (
    final_df_export.groupby(["bank", "host_target"], dropna=False)
    .size()
    .reset_index(name="n_articles")
    .sort_values(["bank", "n_articles"], ascending=[True, False])
)

display(stats)
print("Đã lưu các file vào thư mục riêng của từng ngân hàng.")

Tổng số bài export: 432


,bank,host_target,n_articles
0,BIDV,baochinhphu.vn,100
4,BIDV,thitruongtaichinhtiente.vn,99
5,BIDV,vnexpress.net,97
1,BIDV,cafef.vn,95
3,BIDV,tapchinganhang.gov.vn,25
2,BIDV,fili.vn,16


Đã lưu các file vào thư mục riêng của từng ngân hàng.


## 9) Kiểm tra tiêu đề chứa tên ngân hàng và xuất file (_filter)

In [10]:
# Lọc các bài báo có chứa ít nhất 1 từ khóa của ngân hàng trong tiêu đề
def check_title_contains_bank(title: str, bank_name: str) -> bool:
    if not isinstance(title, str) or not title.strip():
        return False
    
    title_lower = title.lower()
    # Lấy danh sách từ khóa của ngân hàng tương ứng từ dictionary BANKS đã cấu hình ở trên
    aliases = BANKS.get(bank_name, [])
    
    # Kiểm tra xem có bất kỳ từ khóa nào nằm trong title không
    for alias in aliases:
        if alias.lower() in title_lower:
            return True
    return False

# 1. Áp dụng logic lọc vào dataframe cuối (loại bỏ bài có title báo vi phạm hoặc không liên quan)
mask_title = final_df_export.apply(lambda row: check_title_contains_bank(row['title'], row['bank']), axis=1)
filtered_df = final_df_export[mask_title].copy()

print(f"Tổng số bài báo ban đầu: {len(final_df_export)}")
print(f"Số bài báo giữ lại (thỏa mãn tiêu đề có tên NH): {len(filtered_df)}")

# 2 & 3. Xuất Word + Excel cho dữ liệu đã lọc VÀO THƯ MỤC CỦA TỪNG NGÂN HÀNG
print("\n---- XUẤT FILTER THEO TỪNG BANK ----")
for bank_name in filtered_df["bank"].unique():
    bank_filtered_df = filtered_df[filtered_df["bank"] == bank_name]
    
    # Tạo folder: data/2023/BIDV/bronze/
    bank_out_dir = DATA_DIR / bank_name / STAGE_DIRNAME
    bank_out_dir.mkdir(parents=True, exist_ok=True)
    
    out_docx_filter = bank_out_dir / f"{bank_name}_news_gnews_{YEAR}_{TIMESTAMP}_filter.docx"
    out_xlsx_filter = bank_out_dir / f"{bank_name}_news_gnews_{YEAR}_{TIMESTAMP}_filter.xlsx"
    
    export_docx(bank_filtered_df, out_docx_filter)
    bank_filtered_df.to_excel(out_xlsx_filter, index=False)
    
    print(f"[{bank_name} Filtered] DOCX: {out_docx_filter}")
    print(f"[{bank_name} Filtered] XLSX: {out_xlsx_filter}")

Tổng số bài báo ban đầu: 432
Số bài báo giữ lại (thỏa mãn tiêu đề có tên NH): 77

---- XUẤT FILTER THEO TỪNG BANK ----
[BIDV Filtered] DOCX: d:\NCKH\thread_1\data\2023\BIDV\bronze\BIDV_news_gnews_2023_20260404_1606_filter.docx
[BIDV Filtered] XLSX: d:\NCKH\thread_1\data\2023\BIDV\bronze\BIDV_news_gnews_2023_20260404_1606_filter.xlsx
